# Part 1 — Data Exploration & Integrity

**Goal:** understand the three datasets, find quality issues, decide how to handle each one, and produce a clean
loan table for the portfolio & risk analysis (Part 2).

**Outputs**
- `issue_log` — every check that was run, how many rows it hit, and the action taken
- `loans_clean.csv` — de-duplicated loans enriched with merchant fields and analysis flags
- `issue_log.csv` — for the appendix

Snapshot date: **2025-12-31**. Stated coverage: **Jul 2024 – Dec 2025**.

## 0. Setup

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

SNAPSHOT = pd.Timestamp("2025-12-31")
WINDOW_START, WINDOW_END = pd.Timestamp("2024-07-01"), pd.Timestamp("2025-12-31")

# Chart styling: one hue for single-series charts, recessive axes
BLUE, INK, MUTED = "#2a78d6", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.figsize": (10, 3.6), "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
})

### Load data
In Colab: run the cell below; if the CSVs aren't in the session yet it opens an upload dialog —
select `loans.csv`, `users.csv`, `merchants.csv` together. (Alternatively drag them into the Files panel.)

In [ ]:
def find(name):
    hits = sorted(glob.glob(f"**/*{name}*.csv", recursive=True))
    return hits[0] if hits else None

if not all(find(n) for n in ["loans", "users", "merchants"]):
    from google.colab import files  # only needed on Colab
    files.upload()

loans_raw = pd.read_csv(find("loans"))
users = pd.read_csv(find("users"))
merchants = pd.read_csv(find("merchants"))

for name, df in [("loans", loans_raw), ("users", users), ("merchants", merchants)]:
    print(f"{name:10s} {df.shape[0]:>6,} rows x {df.shape[1]} cols")

In [ ]:
# Parse dates with errors='coerce' so malformed values surface as NaT instead of crashing
loans_raw["origination_date"] = pd.to_datetime(loans_raw["origination_date"], errors="coerce")
for c in ["signup_date", "first_loan_date"]:
    users[c] = pd.to_datetime(users[c], errors="coerce")
merchants["integration_date"] = pd.to_datetime(merchants["integration_date"], errors="coerce")

print("unparseable dates:",
      loans_raw["origination_date"].isna().sum(),
      users[["signup_date", "first_loan_date"]].isna().sum().sum(),
      merchants["integration_date"].isna().sum())

## 1. Profile each dataset

### 1a. Loans

In [ ]:
display(loans_raw.head())
display(loans_raw.describe(include="all").T)

In [ ]:
print("Missing values:"); print(loans_raw.isna().sum()[lambda s: s > 0], "\n")
for c in ["product_type", "loan_status", "loan_term_months", "is_new_to_affirm", "cost_of_funds_bps"]:
    print(loans_raw[c].value_counts(dropna=False).to_frame().T, "\n")

In [ ]:
# Product structure: term and APR should line up with product type per the data dictionary
display(pd.crosstab(loans_raw["product_type"], loans_raw["loan_term_months"], margins=True))
display(loans_raw.groupby("product_type")[["apr", "mdr", "loan_amount", "fico_score"]]
        .agg(["min", "median", "max"]).round(4))

### 1b. Users

In [ ]:
display(users.head())
display(users.describe(include="all").T)
print(users["age_bucket"].value_counts(normalize=True).round(3))
print("\nstates:", users["state"].nunique())

### 1c. Merchants

In [ ]:
display(merchants.sort_values(["vertical", "merchant_id"]))
display(merchants.groupby("vertical").agg(merchants=("merchant_id", "count"),
                                          avg_aov=("avg_aov", "mean"),
                                          monthly_checkouts=("avg_monthly_checkouts", "sum"),
                                          promo_merchants=("has_promo_program", "sum")))

## 2. Integrity checks
Every check logs to `issue_log` so nothing gets lost between the notebook and the slides.

In [ ]:
issue_log = []

def log(dataset, check, n_rows, severity, action):
    issue_log.append(dict(dataset=dataset, check=check, rows_affected=int(n_rows),
                          severity=severity, action=action))
    flag = "OK " if n_rows == 0 else "!! "
    print(f"{flag}[{dataset}] {check}: {n_rows:,}")

### 2a. Keys & referential integrity

In [ ]:
L = loans_raw
log("loans", "Duplicate loan_id (extra rows)", L["loan_id"].duplicated().sum(), "High",
    "Investigated below — duplicates are corrupted copies; drop the copy with an out-of-window date")
log("loans", "Fully identical duplicate rows", L.duplicated().sum(), "Info", "None")
log("users", "Duplicate user_id", users["user_id"].duplicated().sum(), "Info", "None")
log("merchants", "Duplicate merchant_id", merchants["merchant_id"].duplicated().sum(), "Info", "None")
log("loans", "user_id not in users", (~L["user_id"].isin(users["user_id"])).sum(), "Info", "None")
log("loans", "merchant_id not in merchants", (~L["merchant_id"].isin(merchants["merchant_id"])).sum(), "Info", "None")
log("users", "Users with no loans", (~users["user_id"].isin(L["user_id"])).sum(), "Info", "None")

### 2b. Dates

In [ ]:
out_of_window = ~L["origination_date"].between(WINDOW_START, WINDOW_END)
print(L.loc[out_of_window, "origination_date"].dt.year.value_counts().sort_index())
log("loans", "origination_date outside Jul-2024..Dec-2025", out_of_window.sum(), "High",
    "Same rows as the duplicate loan_ids — dropped with them")

L_m = L.merge(merchants[["merchant_id", "integration_date"]], on="merchant_id")
log("loans", "Originated before merchant integration_date",
    (L_m["origination_date"] < L_m["integration_date"]).sum(), "Medium", "Symptom of the bad dates; resolved by dedup")
log("users", "first_loan_date before signup_date",
    (users["first_loan_date"] < users["signup_date"]).sum(), "Info", "None")

### 2c. Missing values & domain rules

In [ ]:
log("loans", "Missing fico_score", L["fico_score"].isna().sum(), "Medium",
    "Keep rows; assign 'FICO unknown' band; no imputation (see section 4)")
log("loans", "fico_score outside 300-850", (~L["fico_score"].between(300, 850) & L["fico_score"].notna()).sum(), "Info", "None")

rules = {
    "split_pay with term != 1":            (L["product_type"] == "split_pay") & (L["loan_term_months"] != 1),
    "installment with term not 3/6/12/18": (L["product_type"] != "split_pay") & ~L["loan_term_months"].isin([3, 6, 12, 18]),
    "split_pay / 0pct with APR > 0":       (L["product_type"] != "installment_ib") & (L["apr"] > 0),
    "installment_ib APR outside 10-36%":   (L["product_type"] == "installment_ib") & ~L["apr"].between(0.10, 0.36),
    "loan_amount <= 0":                    L["loan_amount"] <= 0,
    "mdr outside 0-15%":                   ~L["mdr"].between(0, 0.15),
}
for name, mask in rules.items():
    log("loans", name, mask.sum(), "Info", "None")

promo = L.merge(merchants[["merchant_id", "has_promo_program"]], on="merchant_id")
log("loans", "0% APR loan at merchant without promo program",
    ((promo["product_type"] == "installment_0pct") & ~promo["has_promo_program"]).sum(), "Info", "None")

### 2d. Financial reconciliation
If the money fields don't tie out, every unit-economics number downstream is wrong — so test the identities explicitly.

In [ ]:
tol = 0.02
recon = {
    "mdr_fees != loan_amount x mdr":
        (L["mdr_fees"] - L["loan_amount"] * L["mdr"]).abs() > tol,
    "paid_off: total_paid != principal + interest":
        (L["loan_status"] == "paid_off") & ((L["total_paid"] - L["loan_amount"] - L["interest_collected"]).abs() > tol),
    "charged_off: credit_loss != unpaid principal":
        (L["loan_status"] == "charged_off")
        & ((L["credit_loss"] - (L["loan_amount"] - (L["total_paid"] - L["interest_collected"]))).abs() > tol),
    "credit_loss > 0 on paid_off/current":
        L["loan_status"].isin(["paid_off", "current"]) & (L["credit_loss"] > 0),
    "charged_off with zero credit_loss":
        (L["loan_status"] == "charged_off") & (L["credit_loss"] == 0),
    "interest_collected > 0 on 0% APR":
        (L["apr"] == 0) & (L["interest_collected"] > 0),
    "credit_loss > loan_amount":
        L["credit_loss"] > L["loan_amount"],
}
for name, mask in recon.items():
    log("loans", name, mask.sum(), "Info", "None")

### 2e. Cross-table consistency (loans vs. users)

In [ ]:
def cross_table_checks(loans, tag):
    by_user = loans.groupby("user_id").agg(n=("loan_id", "count"), gmv=("loan_amount", "sum"),
                                           first=("origination_date", "min"))
    u = users.set_index("user_id").join(by_user, how="inner")
    new_flags = loans[loans["is_new_to_affirm"]].groupby("user_id").size()
    return {
        f"users.total_loans != loan count ({tag})":            (u["total_loans"] != u["n"]).sum(),
        f"users.total_gmv != sum(loan_amount) ({tag})":        ((u["total_gmv"] - u["gmv"]).abs() > 1).sum(),
        f"users.first_loan_date != min(origination) ({tag})":  (u["first_loan_date"] != u["first"]).sum(),
        f"users with >1 is_new_to_affirm loan ({tag})":        (new_flags > 1).sum(),
    }

for name, n in cross_table_checks(L, "raw").items():
    log("users/loans", name, n, "Medium" if n else "Info", "Symptom of duplicate loans; re-checked after dedup")

## 3. Root cause: the duplicate loans
Four separate-looking problems appeared above: duplicate `loan_id`s, out-of-window dates, and user-level
mismatches (`total_loans`, `first_loan_date`, multiple "new to Affirm" loans). Hypothesis: **they are one problem.**

In [ ]:
dups = L[L["loan_id"].duplicated(keep=False)].sort_values(["loan_id", "origination_date"])
display(dups.head(6))

# Which columns actually differ within a duplicate pair?
differs = {c: (dups.groupby("loan_id")[c].nunique(dropna=False) > 1).sum() for c in L.columns}
print("Columns that differ within duplicate pairs:", {c: n for c, n in differs.items() if n})

In [ ]:
dups = dups.assign(in_window=dups["origination_date"].between(WINDOW_START, WINDOW_END))
per_pair = dups.groupby("loan_id")["in_window"].agg(["size", "sum"])
print("pairs:", len(per_pair), "| every pair has exactly 2 rows:", (per_pair["size"] == 2).all(),
      "| exactly one in-window copy:", (per_pair["sum"] == 1).all())
print("every out-of-window row is a duplicate:", L.loc[out_of_window, "loan_id"].isin(dups["loan_id"]).all())

# Does the bad date look like a typo of the good one (e.g. year swapped)?
pair = dups.pivot_table(index="loan_id", columns="in_window", values="origination_date", aggfunc="first")
pair.columns = ["bad_date", "good_date"]
print("same month-day in both copies:", (pair["bad_date"].dt.strftime("%m-%d") == pair["good_date"].dt.strftime("%m-%d")).mean())

**Finding.** 160 loans appear twice. The two copies are identical in every field *except* `origination_date`;
in each pair exactly one date is inside the stated window and the other is in 1923–1925 or 2031. The bad dates
are not a simple year typo (month/day differ too), so they can't be repaired — but they don't need to be, because
the valid copy exists.

**Action:** drop the 160 out-of-window copies (2.2% of rows). No loan information is lost.

## 4. Clean & re-validate

In [ ]:
loans = L.loc[~out_of_window].copy()
assert loans["loan_id"].is_unique
print(f"{len(L):,} raw rows -> {len(loans):,} clean loans (dropped {len(L) - len(loans)})")

for name, n in cross_table_checks(loans, "after dedup").items():
    log("users/loans", name, n, "Info", "Resolved by dedup")

In [ ]:
# is_new_to_affirm should now match 'first loan of this user' exactly
first_loan_ids = loans.sort_values(["origination_date", "loan_id"]).groupby("user_id")["loan_id"].first()
derived_new = loans["loan_id"].isin(first_loan_ids)
display(pd.crosstab(loans["is_new_to_affirm"], derived_new, rownames=["flag"], colnames=["derived first loan"]))

## 5. Issues that need an assumption (not errors to delete)
These aren't broken rows, but each one changes how the analysis must be done.

### 5a. Missing FICO — is it random?

In [ ]:
loans = loans.merge(merchants[["merchant_id", "vertical", "has_promo_program", "avg_aov"]], on="merchant_id", how="left")
miss = loans["fico_score"].isna()
for c in ["vertical", "product_type", "loan_status", "is_new_to_affirm"]:
    print(miss.groupby(loans[c]).mean().round(3).to_dict())

Missingness is ~6–8% across verticals, products and new/returning users — roughly random. It is *lower* among
charged-off loans (~4%), so missing FICO is not a proxy for risk.
**Decision:** keep these loans in all portfolio totals; put them in a separate **"FICO unknown"** band in
FICO-based cuts rather than imputing (imputation would add false precision to exactly the cut the credit-box
recommendation depends on).

### 5b. `users.fico_at_first_loan` vs. loan-level `fico_score`

In [ ]:
first = loans[loans["is_new_to_affirm"]].merge(users[["user_id", "fico_at_first_loan"]], on="user_id")
diff = (first["fico_score"] - first["fico_at_first_loan"]).abs()
print(f"exact match: {(diff == 0).mean():.1%} | correlation: {first[['fico_score','fico_at_first_loan']].corr().iloc[0,1]:.2f}"
      f" | median abs diff: {diff.median():.0f} pts | max: {diff.max():.0f}")

The two FICO fields disagree for ~99% of first loans (median gap ~20 pts), though they are strongly correlated —
consistent with different pull dates or bureau versions. **Decision:** use loan-level `fico_score` (it is the
score at underwriting for that loan); use `fico_at_first_loan` only if a user-level cut is needed.

### 5c. Maturity / right-censoring — the biggest analytical trap

In [ ]:
loans["orig_month"] = loans["origination_date"].dt.to_period("M")
loans["orig_quarter"] = loans["origination_date"].dt.to_period("Q")
loans["months_on_book"] = ((SNAPSHOT - loans["origination_date"]).dt.days / 30.44).round(1)

status_mix = pd.crosstab(loans["orig_quarter"], loans["loan_status"], normalize="index").round(3)
display(status_mix)

In [ ]:
fig, ax = plt.subplots()
share_current = status_mix["current"] * 100
ax.bar(share_current.index.astype(str), share_current.values, color=BLUE, width=0.6)
for x, v in zip(share_current.index.astype(str), share_current.values):
    ax.text(x, v + 1, f"{v:.0f}%", ha="center", color=MUTED, fontsize=9)
ax.set_title("Share of loans still 'current' by origination quarter")
ax.set_ylabel("% of loans"); ax.set_ylim(0, 100)
plt.tight_layout(); plt.show()

Recent vintages are mostly still `current` — their losses haven't happened yet. A naive "losses ÷ GMV" by segment
will make any segment with younger loans look safer (e.g. anything growing fast, or longer 12/18-month terms).
**Decision:** in Part 2, compare segments **by vintage / months-on-book**, or restrict loss-rate comparisons to
loans that have had time to season (e.g. originated ≤ Jun-2025), and state which view each chart uses.

### 5d. Credit loss on delinquent (not yet charged-off) loans

In [ ]:
st = loans.groupby("loan_status").apply(
    lambda g: pd.Series({"loans": len(g), "loss_$": g["credit_loss"].sum(),
                         "loss_pct_of_amount": (g["credit_loss"] / g["loan_amount"]).mean()}), include_groups=False)
display(st)

`credit_loss` is already positive on delinquent loans (~9% of amount at 30 days, ~31% at 60 days) — the data records a
partial loss before charge-off. **Assumption:** treat `credit_loss` as the realized-to-date loss (as the data
dictionary says) and use it as-is. This is conservative for delinquents that cure and optimistic for `current`
loans that will still default — the vintage view in 5c handles the latter.

### 5e. Cost of funds

In [ ]:
display(pd.crosstab([loans["product_type"], loans["loan_term_months"]], loans["cost_of_funds_bps"]))
print(loans.groupby("orig_quarter")["cost_of_funds_bps"].mean().round(0).to_dict())

`cost_of_funds_bps` ranges 250–450 bps but doesn't vary by product, term, FICO or time — it looks randomly assigned.
**Assumption:** it is an *annualized* rate on the amount funded, so funding cost ≈
`loan_amount × bps/10,000 × term_months/12` (for Split Pay, ~6 weeks). Using the average balance (~½ the principal
for an amortizing loan) is a reasonable refinement; state whichever is used.

### 5f. Business-logic observations worth carrying into Part 2

In [ ]:
mdr_by = loans.groupby(["has_promo_program", "product_type"])["mdr"].mean().unstack().round(4)
display(mdr_by)

big_split = loans[(loans["product_type"] == "split_pay") & (loans["loan_amount"] > 3000)]
print(f"Split Pay loans > $3,000: {len(big_split)} ->", big_split["vertical"].value_counts().to_dict())

loans["sched_maturity"] = loans["origination_date"] + pd.to_timedelta(
    np.where(loans["loan_term_months"] == 1, 42, loans["loan_term_months"] * 30.44), unit="D")
early = loans[loans["loan_status"] == "paid_off"].assign(early=lambda d: d["sched_maturity"] > SNAPSHOT)
print("paid_off before scheduled maturity:", early.groupby("product_type")["early"].mean().round(2).to_dict())

- **0% APR loans do not carry a higher MDR.** The brief says 0% promos are "subsidized by the merchant paying a
  higher MDR", but at promo merchants the average MDR on 0% loans is *lower* than on interest-bearing loans. If that
  holds up, 0% APR is being under-priced — a key input to the "expand 0% APR promos?" question.
- **Large Split Pay tickets** (> $3k, mostly travel/healthcare) are plausible but concentrate exposure in a
  product with MDR-only revenue — flag, don't remove.
- **Early payoff** is common on installment loans, which caps interest income below the contractual APR —
  use `interest_collected`, not APR × amount, when measuring yield.

## 6. Snapshot of the clean portfolio

In [ ]:
monthly = loans.groupby("orig_month")["loan_amount"].sum() / 1e3
fig, ax = plt.subplots()
ax.bar(monthly.index.astype(str), monthly.values, color=BLUE, width=0.7)
ax.set_title("Monthly originations (GMV, $K) — clean data")
ax.set_ylabel("$K"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

print(f"Loans: {len(loans):,} | Users: {loans['user_id'].nunique():,} | GMV: ${loans['loan_amount'].sum():,.0f}")

## 7. Issue log & exports

In [ ]:
issue_df = pd.DataFrame(issue_log)
display(issue_df[issue_df["rows_affected"] > 0])
print(f"\n{(issue_df['rows_affected'] == 0).sum()} checks passed with no issues.")

In [ ]:
loans["fico_band"] = pd.cut(loans["fico_score"], [299, 579, 639, 699, 759, 850],
                            labels=["<580", "580-639", "640-699", "700-759", "760+"]).cat.add_categories("Unknown").fillna("Unknown")
loans["seasoned"] = loans["origination_date"] <= pd.Timestamp("2025-06-30")

loans.to_csv("loans_clean.csv", index=False)
issue_df.to_csv("issue_log.csv", index=False)
print("saved loans_clean.csv", loans.shape, "and issue_log.csv", issue_df.shape)
# On Colab, uncomment to download:
# from google.colab import files; files.download("loans_clean.csv"); files.download("issue_log.csv")

## 8. Summary of assumptions (draft — rewrite in your own words for the deck)
| # | Issue | Rows | Handling / assumption |
|---|---|---|---|
| 1 | Duplicate `loan_id`s whose only difference is a corrupted `origination_date` (1923–25, 2031) | 160 (2.2%) | Drop the out-of-window copy; no information lost. Also resolves the `total_loans`, `first_loan_date`, `is_new_to_affirm` and pre-integration inconsistencies |
| 2 | Missing `fico_score` | ~6.7% | Appears random; keep rows, "Unknown" FICO band, no imputation |
| 3 | `fico_at_first_loan` ≠ loan-level FICO | ~99% of first loans | Use loan-level FICO for underwriting analysis |
| 4 | Young loans haven't had time to default | ~25% `current` | Compare loss by vintage / months-on-book or on seasoned loans only |
| 5 | `credit_loss` recorded on delinquent loans | 236 loans | Use as realized-to-date loss |
| 6 | `cost_of_funds_bps` not linked to product/term/FICO | all | Treat as annual rate × term |
| 7 | 0% APR MDR not higher than interest-bearing MDR | — | Not a data error; a pricing finding for Part 2/3 |